# F2 foreground continuity: final small comparison

A: preserved v0.3 C. B: existing near preservation. C: 70% existing + 30% selective. Geometry, far omission, global transition, exact bottom guard, all small marks and representative trees are fixed. Even the rendered vegetation contribution is identical, isolating terrain continuity.

Install `python -m pip install -e ".[research,dev]"` from the repository root, choose that environment in VSCode/Jupyter, and set `DATASCAPE_DATA_DIR` to the original TIFF folder. Write results to a new directory; preserve earlier studies.

In [ ]:
from pathlib import Path
import os, json
import numpy as np
from IPython.display import display, Image
from datascape.rendering.foreground_continuity import run

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
CONFIG = ROOT / "configs/scenes/seoul_f2_foreground_continuity_v04.json"
assert CONFIG.is_file()
DATA = Path(os.environ.get("DATASCAPE_DATA_DIR", ROOT / "data/raw"))
MASTER = DATA / "SeoulNorth_v02_MASTER_30m.tif"
VEG = DATA / "SeoulNorth_v02_VEG_10m.tif"
assert MASTER.is_file() and VEG.is_file()
OUTPUT = ROOT / "outputs/f2_foreground_continuity_v04"


In [ ]:
result = run(MASTER, VEG, CONFIG, OUTPUT, repo_root=ROOT)
display(Image(filename=str(OUTPUT / "ABC_comparison.png")))
display(Image(filename=str(OUTPUT / "physical_comparison.png")))

## One controlled change

Combine the two existing **local terrain retentions** before multiplying the unchanged global retention. Do not interpolate raw DEM, projection or canopy geometry. The shared v0.3 vegetation contribution stays fixed across all variants.

Ground support is evaluated beneath 42 unique near-root sites, deduplicating the three representatives that share parent roots. Connected image components use a declared ink threshold; they are comparison diagnostics, not evidence of walkability or geographic landform connectivity.

In [ ]:
display(Image(filename=str(OUTPUT / "foreground_detail.png")))
display(Image(filename=str(OUTPUT / "diagnostics.png")))
print(json.dumps(result["record"]["metrics"], indent=2))
inv = result["record"]["invariants"]
assert inv["A_v03_C_pixels_identical"] and inv["F2_baseline_pixels_identical"]
assert inv["outside_near_max_retention_difference"] == 0
assert inv["previous_artifacts_unchanged"] and inv["vegetation_coverage_identical_A_B_C"]
for ink in result["final_inks"].values():
    assert np.all(ink[-2:] == 0)
assert np.all(result["shared_vegetation_ink"][-2:] == 0)
w = result["record"]["config"]["existing_near_weight"]
np.testing.assert_allclose(result["final_inks"]["C"],
                           w * result["final_inks"]["B"] + (1-w) * result["final_inks"]["A"],
                           atol=1e-15)
for key in ["master", "vegetation"]:
    assert result["record"]["source_inputs"][key]["unchanged_after"]
print("Verified one terrain-only combination, fixed vegetation and exact bottom omission.")

## Freeze the selected snapshot for strokes

The configuration records the mix; the archive records the derived terrain retention and unchanged categorical boundary. The common vegetation omission/grammar remains the v0.3 C snapshot. Once visually selected, use the fixed composition for Taebaek curves, terrain structure lines and direction-field strokes, with actual segment omission/splitting/density. No new field generator, projection, tree model, color field or G-code is added here.